# Module 3: Skills + Steering (~8 min)

Two concepts that transform a tool-calling agent into a disciplined analyst:

- **Skills** = trained models carrying domain knowledge. Like an NGS model trained on 10+ years of play data, skills give the agent procedural recipes it can activate situationally
- **Steering** = the broadcast director in the production truck. Deterministic and LLM-based handlers that enforce workflow quality before output reaches the audience

| NGS pattern | strands equivalent | what it does |
| --- | --- | --- |
| trained model carries domain knowledge | `AgentSkills` loads SKILL.md recipes | procedural expertise the agent activates by situation |
| broadcast director gates output | `SteeringHandler` / `LLMSteeringHandler` | enforces quality rules before/after model turns |

In [ ]:
# install dependencies (run once)
%pip install -q strands-agents strands-agents-tools

## Skills — Domain Knowledge Recipes

Each skill is a `SKILL.md` file in a directory under `./skills/`. The format:

```yaml
---
name: skill-name
description: when to activate this skill
---
```

Followed by step-by-step instructions the agent follows when the skill activates.

We have three skills:

| skill | purpose | activates when |
| --- | --- | --- |
| `dynasty-debate` | structured comparison workflow | user compares players or topics |
| `game-breakdown` | game analysis recipe | user asks about a specific game |
| `dynasty-context` | background narrative knowledge | agent needs context tools can't provide |

This is the same pattern as NGS trained models — the skill carries domain expertise the raw LLM doesn't have.

In [ ]:
import sys
sys.path.insert(0, "../shared")
sys.path.insert(0, "../01-agent-loop-tools")

from strands import AgentSkills

# AgentSkills loads every SKILL.md it finds under the given directories
skills_plugin = AgentSkills(skills=["./skills"])

print(f"Skills plugin loaded: {skills_plugin}")
print(f"Skills directory: ./skills/")
print("\nSkill files found:")

import os
for skill_dir in sorted(os.listdir("./skills")):
    skill_path = f"./skills/{skill_dir}/SKILL.md"
    if os.path.exists(skill_path):
        with open(skill_path) as f:
            # read the frontmatter name and description
            lines = f.readlines()
            name = desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
        print(f"  {name}: {desc}")

## Steering — FactCheckHandler (Deterministic)

A `SteeringHandler` runs deterministic logic before/after tool calls or model turns. No LLM involved — just code.

Our `FactCheckHandler` enforces a simple rule: **you must `lookup_player` before you can `get_season_stats` for that player**. This prevents the agent from making statistical claims about players it hasn't verified are on the roster.

The NGS parallel: CloudWatch monitors every inference. The broadcast director doesn't let unchecked stats reach the audience.

In [ ]:
from steering_handlers import FactCheckHandler

fact_checker = FactCheckHandler()

print(f"Handler name: {fact_checker.name}")
print(f"Type: SteeringHandler (deterministic)")
print(f"\nBehavior:")
print(f"  - Hooks into: steer_before_tool")
print(f"  - Monitors: get_season_stats calls")
print(f"  - Rule: player must be verified via lookup_player first")
print(f"  - On violation: returns Guide (redirect agent to look up player)")
print(f"  - On pass: returns Proceed (allow the stats lookup)")

## Steering — DussaultToneHandler (LLM-Based)

An `LLMSteeringHandler` uses a secondary LLM call to evaluate the agent's output. It checks whether the response meets a quality bar — in our case, the Mike Dussault standard:

- Cites specific data (stats, scores, game weeks)
- Connects facts to narrative (why, not just what)
- Avoids vague superlatives ("great", "amazing", "arguably")
- Names what's unknown rather than hedging
- Describes players in terms of team function

The NGS parallel: 90% directional approval from human experts. The LLM-as-judge pattern applied to output quality.

In [ ]:
from steering_handlers import tone_handler, DussaultToneHandler

print(f"Handler name: {tone_handler.name}")
print(f"Type: LLMSteeringHandler (uses a secondary LLM call)")
print(f"\nBehavior:")
print(f"  - Hooks into: steer_after_model")
print(f"  - Evaluates: the agent's response against the Dussault standard")
print(f"  - On pass: Proceed (response meets quality bar)")
print(f"  - On fail: Guide (tells agent what to fix — cite data, avoid vagueness)")
print(f"\nSystem prompt excerpt:")
print(f"  'Check for: specific data citations, narrative connections,")
print(f"   no vague superlatives, name what's unknown, team function framing'")

## Create the Full Agent

Now we wire everything together: tools (data extraction) + skills (domain recipes) + steering (quality enforcement).

The `plugins` list takes all three — AgentSkills, FactCheckHandler, and DussaultToneHandler. The agent gains:
- 5 tools for data lookup
- 3 skills for procedural guidance
- 2 steering handlers (one deterministic, one LLM-based)

In [ ]:
from strands import Agent
from model_provider import get_model
from dussault_tools import (
    lookup_player, get_roster_by_position, get_game_result,
    get_season_stats, get_coaching_staff
)

SYSTEM_PROMPT = """You are Dussault, a 2004 New England Patriots Dussault. Your approach mirrors
the best of patriots.com's coverage — evidence-first, narrative-aware.

When answering:
- Always look up the data before making claims. Never guess stats.
- Connect facts to story — why something happened matters as much as what happened.
- If a question is ambiguous, ask for clarity.
- If the data isn't in your tools, say so clearly rather than fabricating.
- Be specific: cite game weeks, scores, stat lines.
- When comparing players or topics, activate the dynasty-debate skill.
- When analyzing a game, activate the game-breakdown skill."""

agent = Agent(
    model=get_model(),
    tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff],
    plugins=[
        AgentSkills(skills=["./skills"]),
        FactCheckHandler(),
        tone_handler,
    ],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

print("Agent created with:")
print(f"  Tools: lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff")
print(f"  Skills: dynasty-debate, game-breakdown, dynasty-context")
print(f"  Steering: FactCheckHandler (deterministic) + DussaultToneHandler (LLM-based)")

## Test the Fact-Check Guardrail

Watch the steering handlers in action. The agent will:
1. Activate the `dynasty-debate` skill (comparison question)
2. Look up each player before pulling stats (fact-check handler enforces this)
3. Have its response evaluated for Dussault-standard quality (tone handler)

Look for `[FACT-CHECK]` and `[TONE]` log lines — that's the steering handlers firing.

In [ ]:
response = agent("Compare Corey Dillon and Deion Branch — who was more important to the 2004 championship?")
print("\n--- Response complete ---")

## What's Next

The agent now has tools (data), skills (recipes), and steering (quality enforcement). But it forgets everything between sessions.

**Module 4** adds persistent session management — the agent remembers your prior research across restarts. Same pattern as NGS using S3 to store 10+ years of historical play data.

In [ ]:
# try your own queries:
# agent("Break down the Super Bowl for me")
# agent("Was the defense or offense more important?")
# agent("Tell me about the cornerback crisis")